# Lecture 11: Least Squares Problems
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Temperature Anomaly Data: Overfitting vs. Least Squares

5-year averages of worldwide temperature anomaly (NASA, 1955–2000).

In [ ]:
year = np.arange(1955, 2001, 5, dtype=float)
anomaly = np.array([-0.0480, -0.0180, -0.0360, -0.0120, -0.0040,
                     0.1180,  0.2100,  0.3320,  0.3340,  0.4560])

t = year - 1955  # shift for numerical stability
m = len(t)

# Build full Vandermonde matrix (degree m-1 = 9)
A_full = np.column_stack([t**j for j in range(m)])

print(f"Vandermonde matrix: {A_full.shape}")
print(f"cond(A_full) = {np.linalg.cond(A_full):.2e}  (nearly singular!)")

In [ ]:
# Degree 9: interpolation (overfitting)
c_full = np.linalg.solve(A_full, anomaly)

t_fine = np.linspace(0, 45, 500)
p_full = np.polyval(c_full[::-1], t_fine)

plt.figure(figsize=(9, 4.5))
plt.plot(year, anomaly, 'ro', markersize=8, label='Data')
plt.plot(t_fine + 1955, p_full, 'b-', linewidth=1.5, label='Degree 9 (interpolation)')
plt.xlabel('Year')
plt.ylabel('Anomaly (°C)')
plt.title('World temperature anomaly — overfitting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Degree 3: least squares (much better)
A_ls = A_full[:, :4]  # 10 x 4
print(f"Overdetermined system: {A_ls.shape}")
print(f"cond(A_ls) = {np.linalg.cond(A_ls):.2e}\n")

c_ls = np.linalg.lstsq(A_ls, anomaly, rcond=None)[0]
p_ls = np.polyval(c_ls[::-1], t_fine)

plt.figure(figsize=(9, 4.5))
plt.plot(year, anomaly, 'ro', markersize=8, label='Data')
plt.plot(t_fine + 1955, p_ls, 'b-', linewidth=2, label='Degree 3 (least squares)')

# Show residuals
fitted = np.polyval(c_ls[::-1], t)
for yi, fi, xi in zip(anomaly, fitted, year):
    plt.plot([xi, xi], [fi, yi], 'r--', alpha=0.5)

plt.xlabel('Year')
plt.ylabel('Anomaly (°C)')
plt.title('World temperature anomaly — least squares fit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Coefficients: {c_ls}")

## Three Methods for Least Squares

All three give the same answer (for this well-conditioned problem).

In [ ]:
A = A_ls
b = anomaly

# Method 1: Normal equations
c_ne = np.linalg.solve(A.T @ A, A.T @ b)

# Method 2: QR factorization
Q, R = np.linalg.qr(A)
c_qr = np.linalg.solve(R, Q.T @ b)

# Method 3: SVD
U, S, Vt = np.linalg.svd(A, full_matrices=False)
c_svd = Vt.T @ ((U.T @ b) / S)

print("Coefficients from three methods:")
print(f"  Normal eqs: {c_ne}")
print(f"  QR:         {c_qr}")
print(f"  SVD:        {c_svd}")
print(f"\n  Max difference: {max(np.linalg.norm(c_ne - c_qr), np.linalg.norm(c_ne - c_svd)):.2e}")

## The Geometry: Line Fitting Example

Fit $y = c_1 + c_2 t$ through $(0,1)$, $(1,3)$, $(2,2)$.

In [ ]:
A = np.array([[1, 0],
              [1, 1],
              [1, 2]], dtype=float)
b = np.array([1, 3, 2], dtype=float)

# Normal equations
x_star = np.linalg.solve(A.T @ A, A.T @ b)
print(f"x* = {x_star}  (c1 = {x_star[0]:.4f}, c2 = {x_star[1]:.4f})")

# Residual
r = b - A @ x_star
print(f"Residual r = {r}")
print(f"A^T r = {A.T @ r}  (should be zero — orthogonality check)")

In [ ]:
# Plot the fit
t_pts = np.array([0, 1, 2])
t_fine = np.linspace(-0.3, 2.3, 100)
y_fit = x_star[0] + x_star[1] * t_fine

plt.figure(figsize=(7, 5))
plt.plot(t_pts, b, 'ro', markersize=10, zorder=5, label='Data')
plt.plot(t_fine, y_fit, 'b-', linewidth=2,
         label=f'$y = {x_star[0]:.3f} + {x_star[1]:.3f}t$')

# Residuals
fitted = A @ x_star
for ti, bi, fi in zip(t_pts, b, fitted):
    plt.plot([ti, ti], [fi, bi], 'r--', linewidth=1.5, alpha=0.7)
    plt.plot(ti, fi, 'bs', markersize=7)

plt.xlabel('$t$')
plt.ylabel('$y$')
plt.title('Least squares line: residuals are perpendicular to column space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Geometry in $\mathbb{R}^3$: Projection onto Column Space

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

origin = [0, 0, 0]
Ax = A @ x_star  # projection of b

# b
ax.quiver(*origin, *b, color='blue', arrow_length_ratio=0.05, linewidth=2, label='$b$')
# Ax* (projection)
ax.quiver(*origin, *Ax, color='green', arrow_length_ratio=0.07, linewidth=2, label='$Ax^*$ (projection)')
# Residual
ax.quiver(*Ax, *r, color='red', arrow_length_ratio=0.1, linewidth=2, label='$r = b - Ax^*$')

# Column space (plane spanned by columns of A)
s1 = np.linspace(-1, 3, 10)
s2 = np.linspace(-1, 2, 10)
S1, S2 = np.meshgrid(s1, s2)
# Points in the plane: s1 * col1 + s2 * col2
X = S1 * A[0, 0] + S2 * A[0, 1]
Y = S1 * A[1, 0] + S2 * A[1, 1]
Z = S1 * A[2, 0] + S2 * A[2, 1]
ax.plot_surface(X, Y, Z, alpha=0.15, color='green')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$x_3$')
ax.set_title('$b$ projected onto column space of $A$')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## Normal Equations Disaster

When $\kappa(A)$ is large, forming $A^TA$ squares the condition number and destroys the problem.

In [ ]:
eps = 1e-4
A = np.array([[1,    1],
              [eps,  0],
              [0,    eps]])

print(f"A =\n{A}\n")
print(f"cond(A)    = {np.linalg.cond(A):.2e}")
print(f"cond(A^TA) = {np.linalg.cond(A.T @ A):.2e}  (squared!)")
print(f"\nA^TA =\n{A.T @ A}")
print(f"\nIn double precision, the {eps**2:.0e} perturbations are barely visible.")
print(f"For eps=1e-8, they would be wiped out entirely.")

In [ ]:
# Demonstrate with eps = 1e-8: normal equations fail, QR succeeds
np.random.seed(0)
eps = 1e-8
A = np.array([[1,    1],
              [eps,  0],
              [0,    eps]])

x_true = np.array([1.0, 2.0])
b = A @ x_true + 1e-12 * np.random.randn(3)  # tiny noise

# Normal equations
try:
    x_ne = np.linalg.solve(A.T @ A, A.T @ b)
except np.linalg.LinAlgError:
    x_ne = np.full(2, np.nan)

# QR
Q, R = np.linalg.qr(A)
x_qr = np.linalg.solve(R, Q.T @ b)

# SVD
x_svd = np.linalg.lstsq(A, b, rcond=None)[0]

print(f"True x:       {x_true}")
print(f"Normal eqs:   {x_ne}  (error = {np.linalg.norm(x_ne - x_true):.2e})")
print(f"QR:           {x_qr}  (error = {np.linalg.norm(x_qr - x_true):.2e})")
print(f"SVD (lstsq):  {x_svd}  (error = {np.linalg.norm(x_svd - x_true):.2e})")

## Polynomial Fitting: Condition Number vs. Degree

In [ ]:
# Vandermonde condition number grows with polynomial degree
t = np.linspace(0, 1, 50)
degrees = range(1, 25)
conds = []

for d in degrees:
    V = np.column_stack([t**j for j in range(d + 1)])
    conds.append(np.linalg.cond(V))

plt.figure(figsize=(8, 4.5))
plt.semilogy(list(degrees), conds, 'bo-', markersize=4)
plt.axhline(1 / np.finfo(float).eps, color='red', linestyle='--',
            alpha=0.5, label=r'$1/\epsilon_{\mathrm{machine}}$')
plt.xlabel('Polynomial degree')
plt.ylabel(r'$\kappa$(Vandermonde)')
plt.title('Vandermonde matrix condition number vs. degree')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# Noisy data: compare different degrees of polynomial fit
np.random.seed(3)
t_data = np.linspace(0, 1, 100)
y_true = np.sin(4 * t_data)
y_data = y_true + 0.2 * np.random.randn(100)

plt.figure(figsize=(10, 5))
plt.plot(t_data, y_data, '.', alpha=0.4, label='Noisy data')
plt.plot(t_data, y_true, 'k-', alpha=0.3, linewidth=2, label='True function')

t_fine = np.linspace(0, 1, 500)
for deg, color in zip([3, 7, 15], ['blue', 'green', 'red']):
    V = np.column_stack([t_data**j for j in range(deg + 1)])
    c = np.linalg.lstsq(V, y_data, rcond=None)[0]
    V_fine = np.column_stack([t_fine**j for j in range(deg + 1)])
    y_fit = V_fine @ c
    residual = np.linalg.norm(y_data - V @ c)
    plt.plot(t_fine, y_fit, color=color, linewidth=2,
             label=f'Degree {deg} ($\\|r\\|$ = {residual:.3f})')

plt.xlabel('$t$')
plt.ylabel('$y$')
plt.title('Polynomial least squares: effect of degree')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## QR Derivation: Step by Step

In [ ]:
A = np.array([[1, 0],
              [1, 1],
              [1, 2]], dtype=float)
b = np.array([1, 3, 2], dtype=float)

Q, R = np.linalg.qr(A)
print(f"Q_hat =\n{Q.round(6)}\n")
print(f"R_hat =\n{R.round(6)}\n")

# Step 1: Q^T b
z = Q.T @ b
print(f"Q^T b = {z.round(6)}")

# Step 2: Solve R x = z by back-substitution
x = np.linalg.solve(R, z)
print(f"x* = {x.round(6)}")

# Verify
print(f"\nPythagorean theorem:")
print(f"  ||b||^2      = {np.linalg.norm(b)**2:.6f}")
print(f"  ||Q^T b||^2  = {np.linalg.norm(z)**2:.6f}")
print(f"  ||r||^2      = {np.linalg.norm(b - A @ x)**2:.6f}")
print(f"  Sum check:   {np.linalg.norm(z)**2 + np.linalg.norm(b - A @ x)**2:.6f}")

## Rank-Deficient Least Squares

In [ ]:
A = np.array([[1, 1],
              [2, 2],
              [3, 3]], dtype=float)
b = np.array([1, 2, 4], dtype=float)

print(f"A =\n{A}")
print(f"Rank: {np.linalg.matrix_rank(A)}")
print(f"Singular values: {np.linalg.svd(A, compute_uv=False).round(6)}\n")

# Normal equations are singular
print(f"A^TA =\n{A.T @ A}")
print(f"det(A^TA) = {np.linalg.det(A.T @ A):.2e}  (singular!)\n")

# Pseudoinverse gives minimum-norm solution
x_pinv = np.linalg.pinv(A) @ b
print(f"Pseudoinverse solution: x = {x_pinv.round(6)}")
print(f"c1 + c2 = {x_pinv.sum():.6f}")
print(f"c1 == c2: {np.allclose(x_pinv[0], x_pinv[1])}  (minimum norm: equal weight)")
print(f"||x||   = {np.linalg.norm(x_pinv):.6f}")

# Compare: another valid minimizer
x_other = np.array([x_pinv.sum(), 0.0])
print(f"\nAnother minimizer: x = {x_other}")
print(f"||x||   = {np.linalg.norm(x_other):.6f}  (larger norm!)")
print(f"Same Ax: {np.allclose(A @ x_pinv, A @ x_other)}")

## Side-by-Side: Normal Equations vs. QR Accuracy

In [ ]:
np.random.seed(10)
m, n = 80, 10
kappas = np.logspace(1, 12, 20)

err_ne = []
err_qr = []

for kappa in kappas:
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U @ np.diag(s) @ V.T
    x_true = np.random.randn(n)
    b = A @ x_true

    # Normal equations
    try:
        x_ne = np.linalg.solve(A.T @ A, A.T @ b)
    except np.linalg.LinAlgError:
        x_ne = np.full(n, np.nan)
    err_ne.append(np.linalg.norm(x_ne - x_true) / np.linalg.norm(x_true))

    # QR
    Q, R = np.linalg.qr(A)
    x_qr = np.linalg.solve(R, Q.T @ b)
    err_qr.append(np.linalg.norm(x_qr - x_true) / np.linalg.norm(x_true))

eps = np.finfo(float).eps

plt.figure(figsize=(8, 5))
plt.loglog(kappas, err_ne, 'ro-', markersize=5, label='Normal equations')
plt.loglog(kappas, err_qr, 'bs-', markersize=5, label='QR')
plt.loglog(kappas, kappas * eps, 'k--', alpha=0.5, label=r'$\kappa \cdot \epsilon$')
plt.loglog(kappas, kappas**2 * eps, 'k:', alpha=0.5, label=r'$\kappa^2 \cdot \epsilon$')
plt.xlabel(r'$\kappa(A)$')
plt.ylabel('Relative error $\|x - x_{\mathrm{true}}\| / \|x_{\mathrm{true}}\|$')
plt.title('Normal equations vs. QR: accuracy vs. condition number')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()